# Pre-Wave 5 Lab - TravelMind Pre-Flight (no Bedrock credits needed)

You've learned to connect, list models, run Converse, grow it, hit the profile wall, and read errors.
Now **apply all of it** - offline. A simulator stands in for real Bedrock, so this costs nothing.

**How it works**
- Run the **simulator** cell once. It gives you live `bedrock` and `brt` clients (same shapes as real).
- Complete each **TODO** task cell (top to bottom). ~15-20 min total.
- Run the **scoreboard** at the bottom for instant PASS/FAIL on all seven checks.

Rules: edit only the task cells. Don't call real AWS. Aim for 7/7.

In [ ]:
# ===================================================================
#  BEDROCK SIMULATOR  --  DO NOT EDIT.  Run this cell once.
#  It stands in for the real bedrock / bedrock-runtime clients and
#  returns the SAME response shapes. The whole lab runs offline and
#  costs nothing - no credits, no quota, no network.
# ===================================================================
import json

try:
    from botocore.exceptions import ClientError
except Exception:
    class ClientError(Exception): pass

_PREFIXES = ("us.", "eu.", "apac.", "jp.", "global.")
_CATALOG = [
 {"modelId":"anthropic.claude-opus-4-7","modelName":"Claude Opus 4.7","providerName":"Anthropic",
  "inferenceTypesSupported":["ON_DEMAND","INFERENCE_PROFILE"],"inputModalities":["TEXT","IMAGE"],
  "outputModalities":["TEXT"],"responseStreamingSupported":True},
 {"modelId":"anthropic.claude-sonnet-4-6","modelName":"Claude Sonnet 4.6","providerName":"Anthropic",
  "inferenceTypesSupported":["INFERENCE_PROFILE"],"inputModalities":["TEXT","IMAGE"],
  "outputModalities":["TEXT"],"responseStreamingSupported":True},
 {"modelId":"anthropic.claude-haiku-4-5","modelName":"Claude Haiku 4.5","providerName":"Anthropic",
  "inferenceTypesSupported":["INFERENCE_PROFILE"],"inputModalities":["TEXT","IMAGE"],
  "outputModalities":["TEXT"],"responseStreamingSupported":True},
 {"modelId":"amazon.nova-2-lite","modelName":"Nova 2 Lite","providerName":"Amazon",
  "inferenceTypesSupported":["INFERENCE_PROFILE"],"inputModalities":["TEXT","IMAGE"],
  "outputModalities":["TEXT"],"responseStreamingSupported":True},
 {"modelId":"meta.llama3-1-70b-instruct-v1:0","modelName":"Llama 3.1 70B","providerName":"Meta",
  "inferenceTypesSupported":["ON_DEMAND"],"inputModalities":["TEXT"],"outputModalities":["TEXT"],
  "responseStreamingSupported":True},
]
for _m in _CATALOG: _m["modelArn"]="arn:aws:bedrock:us-east-1::foundation-model/"+_m["modelId"]

def _match(b):
    for m in _CATALOG:
        if b==m["modelId"] or b.startswith(m["modelId"]): return m
    return None
def _strip(mid):
    for p in _PREFIXES:
        if mid.startswith(p): return mid[len(p):], True
    return mid, False
def _toks(x): return max(1, len(str(x))//4)
def _last(messages):
    for m in reversed(messages):
        if m["role"]=="user":
            for b in m["content"]:
                if "text" in b: return b["text"]
    return ""
def _reply(messages):
    t=_last(messages).lower()
    if "hello" in t:               return "Hello! Amazon Bedrock runs many models behind one Converse API."
    if "pitch" in t or "goa" in t: return "Sun-soaked Goa in 3 days: beaches by day, night markets after dark."
    if "trip" in t:                return "A 3-day trip works well: two beach days and one for exploring."
    return "Understood. (simulated reply.)"

class _Val(Exception): pass

class _Runtime:
    def _guard(self, modelId):
        base, has_prefix = _strip(modelId); rec=_match(base)
        if (not has_prefix) and rec and "ON_DEMAND" not in rec["inferenceTypesSupported"]:
            raise _Val("An error occurred (ValidationException) when calling the Converse operation: "
                       "Invocation of model ID "+modelId+" with on-demand throughput isn't supported. "
                       "Retry your request with the ID or ARN of an inference profile that contains this model.")
    def converse(self, modelId, messages, system=None, inferenceConfig=None, **kw):
        self._guard(modelId); text=_reply(messages)
        u={"inputTokens":_toks(messages),"outputTokens":_toks(text)}; u["totalTokens"]=u["inputTokens"]+u["outputTokens"]
        return {"output":{"message":{"role":"assistant","content":[{"text":text}]}},"stopReason":"end_turn","usage":u}
    def converse_stream(self, modelId, messages, system=None, inferenceConfig=None, **kw):
        self._guard(modelId); text=_reply(messages)
        def gen():
            for w in text.split(" "): yield {"contentBlockDelta":{"delta":{"text":w+" "}}}
            yield {"messageStop":{"stopReason":"end_turn"}}
        return {"stream":gen()}
    def invoke_model(self, modelId, body, **kw):
        self._guard(modelId); req=json.loads(body)
        class _B:
            def __init__(s,d): s.d=d
            def read(s): return json.dumps(s.d).encode()
        return {"body":_B({"content":[{"type":"text","text":"ok"}],"stop_reason":"end_turn"})}

class _Mgmt:
    def list_foundation_models(self, **kw): return {"modelSummaries":[dict(m) for m in _CATALOG]}
    def get_foundation_model(self, modelIdentifier):
        b,_=_strip(modelIdentifier); return {"modelDetails": dict(_match(b) or _CATALOG[0])}

bedrock = _Mgmt()     # MANAGEMENT client  (list / describe)
brt     = _Runtime()  # RUNTIME client     (converse / stream / invoke)
print("simulator ready - bedrock + brt are live, offline, free")


### Task 1 - Route the call  ·  ~2 min
For each operation, say which client it belongs to (`"bedrock"` or `"bedrock-runtime"`) and the method name.
*Hint: management lists & describes; runtime runs.*

In [ ]:
# Fill each tuple: (client_name, method_name)
ROUTING = {
    "list available models":          ("?", "?"),   # TODO
    "run a prompt and get a reply":    ("?", "?"),   # TODO
    "stream a reply token by token":   ("?", "?"),   # TODO
    "get one model's details":         ("?", "?"),   # TODO
    "send a provider-native raw body": ("?", "?"),   # TODO
}

### Task 2 - Your first Converse call  ·  ~4 min
Complete `first_reply()`: the `content` must be a **list of blocks**, then **walk the response** to pull out
the text, the usage, and the stopReason.

In [ ]:
def first_reply():
    resp = brt.converse(
        modelId="anthropic.claude-opus-4-7",
        system=[{"text": "You are TravelMind's assistant. Be concise."}],
        messages=[{"role": "user", "content": [ None ]}],   # TODO: one text block: {"text": "Say hello and name one Bedrock feature."}
        inferenceConfig={"maxTokens": 200, "temperature": 0.5},
    )
    text  = None   # TODO: walk output > message > content > [0] > text
    usage = None   # TODO
    stop  = None   # TODO: stopReason
    return text, usage, stop

### Task 3 - Pick a model that will actually run  ·  ~4 min
Some models are **profile-only** (the bare ID throws `ValidationException`). Use `get_foundation_model` to read
`inferenceTypesSupported`, then return a `modelId` that runs: bare if `ON_DEMAND`, else prepend `us.`.

In [ ]:
def pick_model_id(base_id):
    info  = bedrock.get_foundation_model(modelIdentifier=base_id)["modelDetails"]
    types = info["inferenceTypesSupported"]
    # TODO: if "ON_DEMAND" in types -> return base_id ; else return "us." + base_id
    raise NotImplementedError("fill this in")

### Task 4a - Multi-turn  ·  ~2 min
Memory is just the messages list. After the first reply, **append the assistant turn**, then **append a
follow-up** asking to make the plan budget-friendly.

In [ ]:
def two_turns():
    messages = [{"role": "user", "content": [{"text": "Plan a 3-day trip."}]}]
    r1 = brt.converse(modelId="anthropic.claude-opus-4-7", messages=messages, inferenceConfig={"maxTokens": 200})
    # TODO: append r1's assistant message, then append a user follow-up mentioning "budget"
    r2 = brt.converse(modelId="anthropic.claude-opus-4-7", messages=messages, inferenceConfig={"maxTokens": 200})
    return messages

### Task 4b - Robust JSON  ·  ~3 min
Models hand you broken or fenced JSON eventually. Make `safe_json` strip ` ```json ` fences, try to parse,
and on any failure return `{}` - **never raise**.

In [ ]:
def safe_json(text):
    import json
    # TODO: strip ```json / ``` fences if present, then try json.loads; on failure return {}
    raise NotImplementedError("fill this in")

### Task 5 - Stream it  ·  ~3 min
Complete `stream_text`: iterate the event stream and assemble every `contentBlockDelta` text piece into one
string.

In [ ]:
def stream_text(prompt):
    out = []
    stream = brt.converse_stream(
        modelId="anthropic.claude-opus-4-7",
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 200},
    )
    # TODO: for each chunk in stream["stream"], if it has "contentBlockDelta", append the delta text
    return "".join(out).strip()

### Task 6 - Diagnose the error  ·  ~3 min
Implement `diagnose(error_text)` returning `(is_account_issue, fix)`.
`is_account_issue` is **True only when the fix lives in AWS, not your code** (the throttle you hit).
Cover: *too many tokens per day*, *on-demand throughput isn't supported*, *unable to locate credentials*.

In [ ]:
def diagnose(error_text):
    e = error_text.lower()
    # TODO: return (is_account_issue: bool, fix: str) for each of the three error classes
    raise NotImplementedError("fill this in")

In [ ]:
# ===================== GRADERS  (DO NOT EDIT) =====================
def grade_routing(R):
    pairs={"bedrock":bedrock,"bedrock-runtime":brt}
    for op,(cli,meth) in R.items():
        assert cli in pairs, f"{op}: client must be 'bedrock' or 'bedrock-runtime'"
        obj=pairs[cli]; other=brt if cli=="bedrock" else bedrock
        assert hasattr(obj,meth), f"{op}: '{cli}' has no method '{meth}'"
        assert not hasattr(other,meth), f"{op}: '{meth}' lives on the OTHER client"
    return True

def grade_t2():
    t,u,s=first_reply()
    assert isinstance(t,str) and len(t)>0, "text should be the reply string"
    assert {"inputTokens","outputTokens","totalTokens"} <= set(u), "usage must have the token fields"
    assert s=="end_turn", "stopReason should be 'end_turn'"
    return True

def grade_t3():
    hid=pick_model_id("anthropic.claude-haiku-4-5")
    brt.converse(modelId=hid, messages=[{"role":"user","content":[{"text":"hi"}]}], inferenceConfig={"maxTokens":20})
    assert hid.startswith("us."), "Haiku is profile-only -> needs the us. prefix"
    oid=pick_model_id("anthropic.claude-opus-4-7")
    assert oid=="anthropic.claude-opus-4-7", "Opus 4.7 supports on-demand -> no prefix"
    return True

def grade_t4a():
    m=two_turns()
    assert [x["role"] for x in m]==["user","assistant","user"], "roles must be user, assistant, user"
    assert "budget" in m[-1]["content"][0]["text"].lower(), "the follow-up should ask for budget-friendly"
    return True

def grade_t4b():
    assert safe_json('{"a": 1}')=={"a":1}, "clean JSON should parse"
    assert safe_json('```json\n{"b": 2}\n```')=={"b":2}, "fenced JSON should parse after stripping fences"
    assert safe_json('sorry, here you go')=={}, "garbage should return {} not raise"
    return True

def grade_t5():
    s=stream_text("Write a 2-line pitch for Goa.")
    ref=brt.converse(modelId="anthropic.claude-opus-4-7",
        messages=[{"role":"user","content":[{"text":"Write a 2-line pitch for Goa."}]}],
        inferenceConfig={"maxTokens":200})["output"]["message"]["content"][0]["text"]
    assert s==ref, "the assembled stream should equal the full reply"
    return True

def grade_t6():
    a,f=diagnose("ThrottlingException: Too many tokens per day, please wait before trying again")
    assert a is True and ("quota" in f.lower() or "support" in f.lower()), "per-day throttle = account quota issue"
    a,f=diagnose("ValidationException: on-demand throughput isn't supported")
    assert a is False and any(k in f.lower() for k in ("profile","prefix","us.")), "fix = inference profile / us. prefix"
    a,f=diagnose("Unable to locate credentials")
    assert a is False and any(k in f.lower() for k in ("bearer","configure","token")), "fix = set credentials"
    return True
print("graders loaded")


In [ ]:
# ===================== SCOREBOARD - run me last =====================
TASKS=[("1  Route the call",      lambda: grade_routing(ROUTING)),
       ("2  First Converse call", grade_t2),
       ("3  Pick a runnable id",  grade_t3),
       ("4a Multi-turn",          grade_t4a),
       ("4b Robust JSON",         grade_t4b),
       ("5  Streaming",           grade_t5),
       ("6  Diagnose errors",     grade_t6)]
passed=0
for name,fn in TASKS:
    try:
        fn(); print(f"PASS   {name}"); passed+=1
    except Exception as e:
        print(f"FAIL   {name}  ->  {e}")
print(f"\n>>> {passed}/{len(TASKS)} passing")
if passed==len(TASKS): print(">>> You are ready for Day 5.")
